In [41]:
import os
import glob
from google import genai
import re
import shutil
import unicodedata
import time
from dotenv import load_dotenv
from xai_sdk import Client
from xai_sdk.chat import user, system
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter, SpacyTextSplitter
import pdfplumber
from google.genai import types
import json
import asyncio
import uuid

In [2]:
MODEL = "grok-4.20-0309-reasoning"
GEMINI_MODEL = "gemini-3.1-flash-lite"
book_ratio = 50 #What % of the book has to be left

In [3]:
def get_paths():
    pdf_list = glob.glob("input/*.pdf")
    pdf_list = [re.sub(r'\\', '/', pdf) for pdf in pdf_list]
    print(pdf_list[0])
    return pdf_list

In [4]:
def extract_text(pdf_path):
    """Extract all text from a PDF. Returns (full_text, page_count)."""
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        page_count = len(pdf.pages)
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                pages.append(text)
    return "\n\n".join(pages), page_count

In [5]:
pdf_list = glob.glob("input/*.pdf")
pdf = pdf_list[0]
text, pageCount = extract_text(pdf)
print(text, pageCount)

Jonas Biliūnas
LIŪDNA PASAKA
BALTASAI ŠEŠĖLIS
Prakalbos vietoje
O jaunosios dienos mano! Kaip melsvam ore gervės nykstat jūs, tiktai ką pasirodžiusios... Žiūriu į jus
pralėkusias, kaip į sapną gražų, ir matau tik, kad jau artinas ruduo gyvenimo mano.
O Ramūta! Kodėl nepažinau tavęs, dar mažutis, nekaltas būdamas! Tada tokios gražios, laimingos
buvo dienos. Būtume, už rankų susitvėrę, Šventosios pakrančiais vaikščioję, lakštingalos giesmių klausę.
Būčiau tave po miškus ir krūmus išvedžiojęs, paukščių lizdus parodęs, išsirpusių uogų parinkęs. Bet tu
buvai toli nuo manęs, toli, ir aš net nežinojau, kad tu gyveni pasaulyje...
Taip, nežinojau... Bet jaučiau... Nuo pat mažų dienų tavo paveikslas mano širdyje gyveno. O kai,
mokiniu būdamas, traukiniu gimnazijon važiuodavau ir tavo tėvynės mirguojančius klonius ir laukus ir
gimtąjį sodžių pro langą išvysdavau, mano krūtinėje netikėtai sujudėdavo naujas, geras jautimas: kaip
paukštis pro šalį pralėkdavau, bet ilgai dar į išsitiesusį paveiksląži

In [6]:
print(len(text))

46612


In [7]:
print(len(text)/20)

2330.6


In [8]:
def chunk_book_for_gemini(text):
    chunksize = 50000 if len(text) <= 600000 else len(text)/20

    chunker = RecursiveCharacterTextSplitter(chunk_size=chunksize, chunk_overlap=200)
    chunks = chunker.split_text(text)
    print(f"Chunks: {len(chunks)}")

    return chunks


In [9]:
def chunk_book_for_grok(text):
    chunker = RecursiveCharacterTextSplitter(chunk_size=11600, chunk_overlap=200)
    chunks = chunker.split_text(text)
    print(f"Chunks: {len(chunks)}")

    return chunks


In [10]:
def convert_pdfs_to_ascii():
    pdf_list = glob.glob("input/*.pdf")
    pdf_list = [re.sub(r'\\', '/', pdf) for pdf in pdf_list]

    for pdf in pdf_list:
        mod_pdf = pdf.replace("input", "ascii_input")
        if os.path.exists(mod_pdf):
            print(f"{mod_pdf} already exists")
        else:
            base = os.path.basename(pdf)
            ascii_base = unicodedata.normalize("NFKD", base).encode("ascii", "ignore").decode("ascii")
            ascii_base = re.sub(r"[^A-Za-z0-9._-]+", "_", ascii_base)
            safe_path = os.path.join("ascii_input", ascii_base)
            # Keep original file; just make a copy with safe name
            shutil.copy2(pdf, safe_path)
            print(f"{mod_pdf} created")
    return None

In [11]:
def upload_pdfs_to_gemini():
    client = genai.Client()
    pdfs=glob.glob("ascii_input/*.pdf")
    file_pdf_list = []

    # Upload each PDF
    for pdf in pdfs:
        myfile = client.files.upload(file=pdf)
        file_name = myfile.name
        myfile = client.files.get(name=file_name)
        print(myfile)
        file_pdf_list.append({"book_name": os.path.basename(pdf), "file": myfile})

        # Wait for the file to be active
        while myfile.state.name != "ACTIVE":
            time.sleep(1)
            myfile = client.files.get(name=file_name)
            print(myfile)

    return file_pdf_list

In [12]:
def upload_pdfs_to_grok():
    pdfs=glob.glob("ascii_input/*.pdf")
    grok_file_pdf_list = []

    # Upload each PDF
    load_dotenv()
    client = Client(api_key=os.getenv("XAI_API_KEY"))
    for pdf in pdfs:
        file = client.files.upload(pdf, expires_after=82800) #expires after 23 hours
        grok_file_pdf_list.append({"book_name": os.path.basename(pdf), "file": file})



    return grok_file_pdf_list

In [13]:

def call_gemini_batch(chunks, gemini_file, ratio_pct=book_ratio):
    load_dotenv()
    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    model = "gemini-2.5-flash"
    
    file = gemini_file["file"]
    fileID = file.name
    book_name = gemini_file["book_name"]
    if ratio_pct is None:
        ratio_pct = book_ratio

    batch_filename = "gemini_batch_input.jsonl"
    
    with open(batch_filename, 'w', encoding='utf-8') as f:
        for i, chunk_text in enumerate(chunks):
            chunk_num = i + 1
            target_len = int(len(chunk_text) * ratio_pct / 100)
            
            system_text = GEMINI_SYSTEM.format(ratio_pct=ratio_pct)
            user_text = GEMINI_USER.format(
                ratio_pct=ratio_pct,
                chunk_num=chunk_num,
                target_len=target_len,
                lo_len=int(target_len * 0.90),
                hi_len=int(target_len * 1.1),
                chunk_start=chunk_text[:500],
                chunk_end=chunk_text[-500:],
                file_id=fileID
            )

            request_data = {
                "key": f"{book_name}_{chunk_num:03d}",
                "request": {
                    "method": "generateContent",
                    "model": f"models/{model}",
                    "system_instruction": {"parts": [{"text": system_text}]},
                    "contents": [{
                        "role": "user",
                        "parts": [
                            {"file_data": {"file_uri": file.uri, "mime_type": "application/pdf"}},
                            {"text": user_text}
                        ]
                    }]
                }
            }
            f.write(json.dumps(request_data) + '\n')

    print(f"Uploading batch instructions...")
    batch_input_file = client.files.upload(
        file=batch_filename,
        config={"mime_type": "application/jsonl"}
    )

    print("creating a batch")
    batch_job = client.batches.create(
        model=model,
        src=batch_input_file.name,
        config={'display_name': f'book-compression-{chunk_num}-chunks'}
    )

    print(f"Batch Job Created: {batch_job.name}  state: {batch_job.state}")
    os.remove(batch_filename)
    return batch_job

In [14]:
load_dotenv()
client = Client(api_key=os.getenv("XAI_API_KEY"))

In [20]:
GROK_SYSTEM = """\
Tu esi lietuvių literatūros teksto atkūrimo redaktorius.

<constraints>
MUST: Naudok tik originalaus failo ir Gemini versijos turinį.
MUST: Galutinio teksto ilgis — ±10% Gemini versijos ilgio.
MUST: Išlaik autoriaus sakinių ritmą, leksiką ir pasakojimo toną.
MUST: Atkurk viską, kas prarasta — vardus, vietas, datas, dialogo fragmentus, emocines reakcijas, aplinkos detales.
NEVER: Nepridėk informacijos, kurios nėra originale.
NEVER: Grąžink tik tekstą — jokių komentarų, įžangų, paaiškinimų.
NEVER: Nerašyk AI stiliumi — tekstas turi skambėti kaip autorius.
</constraints>

<task>
Vienas praėjimas: surask, ko trūksta Gemini versijoje lyginant su originalu, įterpk į Gemini struktūrą, ištaisyk nenatūralią lietuvių kalbą. Jei reikia jungties su ankstesniu fragmentu — maksimaliai 1–2 frazės iš originalo turinio. Grąžink tik rezultatą.
</task>

<reasoning_discipline>
NEVER: Necituok ir nerašyk pilno teksto reasoning metu.
MUST: Reasoning — tik trūkstamų elementų sąrašas trumpais įrašais (pvz.: "vardas X → įterpti 3 par.", "dialogas → atkurti"). Jokio teksto perrašymo.
MUST: Kuo mažiau reasoning — kuo daugiau tiesiogiai į rezultatą.
</reasoning_discipline>
"""

GROK_USER = """\
<original_file.id>
{file}
</original_file.id>

<gemini_compressed_version>
{compressed}
</gemini_compressed_version>

Atkurk ir patobulink sutrumpintą fragmentą remdamasis originalu. Pateik tik galutinį tekstą.
Galutinis tekstas:\
"""

In [21]:
GEMINI_SYSTEM = """\
Tu esi aukštos kvalifikacijos lietuvių literatūros redaktorius ir teksto trumpintojas.
Tavo užduotis yra sutrumpinti pateiktą knygos fragmentą TIKSLIAI iki {ratio_pct}% jo originalaus simbolių skaičiaus.

SVARBIOS TAISYKLĖS:
1. ILGIO REIKALAVIMAS — PRIVALOMAS: rezultato tekstas turi būti {ratio_pct}% originalaus fragmento ilgio (±5%).
2. Išlaik VISUS esminius siužeto įvykius, pagrindinių veikėjų vystymąsi ir svarbias scenas.
3. Išlaik autoriaus kalbos stilių ir toną kiek įmanoma.
4. Sumažink antrinius aprašymus, pasikartojančias mintis ir per ilgus dialogus.
5. Trumpink, bet NEKURK naujų faktų ar įvykių — tik rinktinai šalink.
6. Rezultatas turi būti sklandžiai skaitomas lietuviškas tekstas.
7. NEANALIZUOK ir NESKAIČIUOK — tiesiog pateik sutrumpintą tekstą be jokių komentarų, skaičiavimų ar įžangų.
8. Nepridėk jokių antraščių ar paaiškinimų — pradėk tiesiogiai nuo teksto.\
"""

GEMINI_USER = """\
Sutrumpink toliau pateiktą knygos fragmentą iki {ratio_pct}% jo dydžio.

Tikslinis ilgis: {target_len} simbolių (leistinas diapazonas: {lo_len}–{hi_len}).

--- ORGINALAUS TEKSTO FRAGMENTAS ---
{chunk}
--- ORGINALAUS TEKSTO FRAGMENTO PABAIGA ---

Pateik tik sutrumpintą fragmento tekstą. Jokių komentarų ar skaičiavimų.
Galutinis tekstas:\
"""

In [22]:
def make_grok_prompt(file_id, gemini_chunk):
    user_prompt = user(GROK_USER.format(file=file_id, compressed=gemini_chunk))
    system_prompt = system(GROK_SYSTEM)

    return system_prompt, user_prompt

In [42]:
def make_batch(model, gemini_chunk, file_id, batch_id, request_index):
    system_prompt, user_prompt = make_grok_prompt(file_id, gemini_chunk)

    chat = client.chat.create(
        model=model,
        batch_request_id = f"{batch_id}-{request_index}"
    )
    chat.append(system_prompt)
    chat.append(user_prompt)

    return chat

In [ ]:
'''
def make_request(book_name, file_id, gemini_chunks):
    batch_requests = []

    batch = client.batch.create(batch_name=book_name)
    batch_id = batch.batch_id

    for chunk in gemini_chunks:
        chunkBatch = make_batch(MODEL, chunk, file_id, batch_id)
        batch_requests.append(chunkBatch)
    
    client.batch.add(batch_id=batch_id, batch_requests=batch_requests)

    return batch_id
    '''

In [50]:
def make_request(book_name, file_id, gemini_chunks):
    batch = client.batch.create(batch_name=book_name)
    batch_id = batch.batch_id
    batch_requests = []

    for i, chunk in enumerate(start=1, iterable=gemini_chunks):
        chunkBatch = make_batch(MODEL, chunk, file_id, batch_id, request_index=i)
        batch_requests.append(chunkBatch)

    client.batch.add(batch_id=batch_id, batch_requests=batch_requests)
    return batch_id

In [51]:
def convert_books_gemini():
    batch_list = []

    print("uploading files to files api")
    geminiFiles = upload_pdfs_to_gemini()
    text, pageCount = extract_text(pdf)
    chunks = chunk_book_for_gemini(text)

    #Gemini shortening
    for file in geminiFiles:
        print(f"Trumpinamas failas: {file}")
        batch_list.append(call_gemini_batch(chunks, file))

    return batch_list

In [27]:
load_dotenv(override=True)
gemini = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

def gemini_call(chunk):
    target = len(chunk) * 50 / 100
    lo = target*0.9
    hi=target*1.1
    system = GEMINI_SYSTEM.format(ratio_pct=book_ratio)
    user = GEMINI_USER.format(chunk=chunk, ratio_pct=book_ratio, target_len=target, lo_len=lo, hi_len=hi)
    print(user)

    response = gemini.models.generate_content(
        model=GEMINI_MODEL,
        contents=[
            {"role": "user", "parts": [{"text": user}]}
        ],
        config={"system_instruction": system}
    )
    return response




In [28]:
def shorten_gemini():
    files = glob.glob("ascii_input/*.pdf")
    gemini_books = []

    for file in files:
        converted = ""

        text, pageCount = extract_text(file)
        chunks = chunk_book_for_gemini(text)

        for chunk in chunks:
            response = gemini_call(chunk)
            converted += "\n"
            converted += response.text

        gemini_books.append(converted)
    
    return gemini_books

In [ ]:
def convert_books_grok():
    grokFiles = upload_pdfs_to_grok()
    gemini_books = shorten_gemini()
    gemini_chunks = []

    for i in range(len(gemini_books)):
        gemini_chunks.append(chunk_book_for_grok(gemini_books[i]))

    batchIds = []

    for i, file in enumerate(grokFiles):
        bookName = file["book_name"]
        file_id = file["file"].name
        batchIds.append(make_request(file_id, gemini_chunks[i]["content"]))

    return batchIds

Chunks: 1
Sutrumpink toliau pateiktą knygos fragmentą iki 50% jo dydžio.

Tikslinis ilgis: 23306.0 simbolių (leistinas diapazonas: 20975.4–25636.600000000002).

--- ORGINALAUS TEKSTO FRAGMENTAS ---
Jonas Biliūnas
LIŪDNA PASAKA
BALTASAI ŠEŠĖLIS
Prakalbos vietoje
O jaunosios dienos mano! Kaip melsvam ore gervės nykstat jūs, tiktai ką pasirodžiusios... Žiūriu į jus
pralėkusias, kaip į sapną gražų, ir matau tik, kad jau artinas ruduo gyvenimo mano.
O Ramūta! Kodėl nepažinau tavęs, dar mažutis, nekaltas būdamas! Tada tokios gražios, laimingos
buvo dienos. Būtume, už rankų susitvėrę, Šventosios pakrančiais vaikščioję, lakštingalos giesmių klausę.
Būčiau tave po miškus ir krūmus išvedžiojęs, paukščių lizdus parodęs, išsirpusių uogų parinkęs. Bet tu
buvai toli nuo manęs, toli, ir aš net nežinojau, kad tu gyveni pasaulyje...
Taip, nežinojau... Bet jaučiau... Nuo pat mažų dienų tavo paveikslas mano širdyje gyveno. O kai,
mokiniu būdamas, traukiniu gimnazijon važiuodavau ir tavo tėvynės mirguojan

KeyboardInterrupt: 

In [29]:
gemini_books = shorten_gemini()

Chunks: 1
Sutrumpink toliau pateiktą knygos fragmentą iki 50% jo dydžio.

Tikslinis ilgis: 23306.0 simbolių (leistinas diapazonas: 20975.4–25636.600000000002).

--- ORGINALAUS TEKSTO FRAGMENTAS ---
Jonas Biliūnas
LIŪDNA PASAKA
BALTASAI ŠEŠĖLIS
Prakalbos vietoje
O jaunosios dienos mano! Kaip melsvam ore gervės nykstat jūs, tiktai ką pasirodžiusios... Žiūriu į jus
pralėkusias, kaip į sapną gražų, ir matau tik, kad jau artinas ruduo gyvenimo mano.
O Ramūta! Kodėl nepažinau tavęs, dar mažutis, nekaltas būdamas! Tada tokios gražios, laimingos
buvo dienos. Būtume, už rankų susitvėrę, Šventosios pakrančiais vaikščioję, lakštingalos giesmių klausę.
Būčiau tave po miškus ir krūmus išvedžiojęs, paukščių lizdus parodęs, išsirpusių uogų parinkęs. Bet tu
buvai toli nuo manęs, toli, ir aš net nežinojau, kad tu gyveni pasaulyje...
Taip, nežinojau... Bet jaučiau... Nuo pat mažų dienų tavo paveikslas mano širdyje gyveno. O kai,
mokiniu būdamas, traukiniu gimnazijon važiuodavau ir tavo tėvynės mirguojan

In [31]:
grokFiles = upload_pdfs_to_grok()

In [32]:
gemini_chunks = []

for i in range(len(gemini_books)):
    gemini_chunks.append(chunk_book_for_grok(gemini_books[i]))

Chunks: 1


In [52]:
batchIds = []

for i, file in enumerate(grokFiles):
    bookName = file["book_name"]
    file_id = file["file"].id
    batchIds.append(make_request(bookName, file_id, gemini_chunks[i][0]))

In [62]:
response = client.batch.list(limit=20)
for batch in response.batches:
    status = "complete" if batch.state.num_pending == 0 else "processing"
    print(f"{batch.name} ({batch.batch_id}): {status}")

Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_91f90d27-a310-4e67-9e7f-975e7fa306bd): processing
Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf (batch_2ba0a24b-dbb0-4971-b11d-362d6fca8ef6): complete


In [63]:
print(batchIds)

['batch_91f90d27-a310-4e67-9e7f-975e7fa306bd']


In [59]:
def wait_for_grok_batch(batchID):
    load_dotenv()
    client = Client(api_key=os.getenv("XAI_API_KEY"))
    # Paginate through all results
    all_succeeded = []
    all_failed = []
    pagination_token = None
    while True:
        # Fetch a page of results (limit controls page size)
        page = client.batch.list_batch_results(
            batch_id=batchID,
            limit=100,
            pagination_token=pagination_token,
        )
        
        # Collect results from this page
        all_succeeded.extend(page.succeeded)
        all_failed.extend(page.failed)
        
        # Check if there are more pages
        if page.pagination_token is None:
            break
        pagination_token = page.pagination_token
    # Process results - handle different response types
    print(f"Successfully processed: {len(all_succeeded)} requests")
    for result in all_succeeded:
        rid = result.batch_request_id
        resp = result.proto.response
        if resp.HasField("completion_response"):
            # Chat completion response
            print(f"[{rid}] {result.response.content}")
            print(f"  Tokens used: {result.response.usage.total_tokens}")
    if all_failed:
        print(f"\nFailed: {len(all_failed)} requests")
        for result in all_failed:
            print(f"[{result.batch_request_id}] Error: {result.error_message}")

    return all_succeeded, all_failed

In [71]:
# ── STEP 1: Convert input PDFs to ASCII-safe filenames ──────────────────────
convert_pdfs_to_ascii()

ascii_input/Jonas_Biliūnas._tik_Liūdna_pasaka.LG1800.pdf created


In [ ]:
# ── STEP 2: Upload PDFs to Gemini and submit compression batch jobs ──────────
# Returns a list of Gemini batch job objects (one per book)
batch_list = convert_books_gemini()

uploading files to files api
name='files/o2tr9b5cxlg6' display_name=None mime_type='application/pdf' size_bytes=230204 create_time=datetime.datetime(2026, 5, 10, 9, 3, 50, 341517, tzinfo=TzInfo(0)) expiration_time=datetime.datetime(2026, 5, 12, 9, 3, 49, 741223, tzinfo=TzInfo(0)) update_time=datetime.datetime(2026, 5, 10, 9, 3, 50, 341517, tzinfo=TzInfo(0)) sha256_hash='NTM1MzlkYjk4MWM2OWYzYTQ0YjYzOTNhODZlMmZjNDIxNGVkYmIwYjhhNjliMTJjNDYwZTY2YzM4OTE0ZGU5MA==' uri='https://generativelanguage.googleapis.com/v1beta/files/o2tr9b5cxlg6' download_uri=None state=<FileState.ACTIVE: 'ACTIVE'> source=<FileSource.UPLOADED: 'UPLOADED'> video_metadata=None error=None
Chunks: 5
Trumpinamas failas: {'book_name': 'Jonas_Biliunas._tik_Liudna_pasaka.LG1800.pdf', 'file': File(
  create_time=datetime.datetime(2026, 5, 10, 9, 3, 50, 341517, tzinfo=TzInfo(0)),
  expiration_time=datetime.datetime(2026, 5, 12, 9, 3, 49, 741223, tzinfo=TzInfo(0)),
  mime_type='application/pdf',
  name='files/o2tr9b5cxlg6',
  sh

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource has been exhausted (e.g. check quota).', 'status': 'RESOURCE_EXHAUSTED'}}

In [58]:
#Delete files
client = genai.Client()

print('My files:')
for f in client.files.list():
    print(' ', f.name)
    client.files.delete(name=f.name)

My files:


In [ ]:
#Check batches
client = genai.Client()

print('My files:')
for f in client.batches.list():
    print(f"ID: {f.name}, State: {f.state}")
    

My files:


In [72]:
# ── STEP 3: Wait for Gemini results, then submit to Grok for restoration ─────
# Returns a list of Grok batch IDs (one per book)
batchIds = convert_books_grok()


Chunks: 1
Sutrumpink toliau pateiktą knygos fragmentą iki 50% jo dydžio.

Tikslinis ilgis: 23306.0 simbolių (leistinas diapazonas: 20975.4–25636.600000000002).

--- ORGINALAUS TEKSTO FRAGMENTAS ---
Jonas Biliūnas
LIŪDNA PASAKA
BALTASAI ŠEŠĖLIS
Prakalbos vietoje
O jaunosios dienos mano! Kaip melsvam ore gervės nykstat jūs, tiktai ką pasirodžiusios... Žiūriu į jus
pralėkusias, kaip į sapną gražų, ir matau tik, kad jau artinas ruduo gyvenimo mano.
O Ramūta! Kodėl nepažinau tavęs, dar mažutis, nekaltas būdamas! Tada tokios gražios, laimingos
buvo dienos. Būtume, už rankų susitvėrę, Šventosios pakrančiais vaikščioję, lakštingalos giesmių klausę.
Būčiau tave po miškus ir krūmus išvedžiojęs, paukščių lizdus parodęs, išsirpusių uogų parinkęs. Bet tu
buvai toli nuo manęs, toli, ir aš net nežinojau, kad tu gyveni pasaulyje...
Taip, nežinojau... Bet jaučiau... Nuo pat mažų dienų tavo paveikslas mano širdyje gyveno. O kai,
mokiniu būdamas, traukiniu gimnazijon važiuodavau ir tavo tėvynės mirguojan

AttributeError: name

In [65]:
cancelled_batch = client.batch.cancel(batch_id="batch_91f90d27-a310-4e67-9e7f-975e7fa306bd")
print(f"Cancelled batch: {cancelled_batch.batch_id}")
print(f"Completed before cancellation: {cancelled_batch.state.num_success} requests")

Cancelled batch: batch_91f90d27-a310-4e67-9e7f-975e7fa306bd
Completed before cancellation: 285 requests


In [64]:
# ── STEP 4: Wait for Grok results and print them ─────────────────────────────
#for batchId in batchIds:
succeeded, failed = wait_for_grok_batch("batch_91f90d27-a310-4e67-9e7f-975e7fa306bd")
print(f"\nDone — {len(succeeded)} succeeded, {len(failed)} failed")

Successfully processed: 245 requests
[batch_91f90d27-a310-4e67-9e7f-975e7fa306bd-1] O
  Tokens used: 2584
[batch_91f90d27-a310-4e67-9e7f-975e7fa306bd-10] o
  Tokens used: 2774
[batch_91f90d27-a310-4e67-9e7f-975e7fa306bd-100] Buvo šalta rudens popietė, kai senasis dvaro kambarys prisipildė dulkių ir tylos. Jonas sėdėjo prie lango, pirštais barbendamas į medinį stalą, o jo akys klaidžiojo po apleistą sodą, kur obelys jau seniai metė lapus. „Ar prisimeni tą vasarą, kai viskas dar buvo kitaip?“ – staiga paklausė jis, balsas skambėjo kimiai, lyg būtų bijojęs atsakymo. Elena stovėjo tarpduryje, rankose laikydama seną laišką, jos veidas išblyško, o akyse šmėstelėjo tas pats senas skausmas, kurį abu stengėsi pamiršti. Lauke pradėjo lyti, lietaus lašai barbeno į stiklą kaip priminimas apie prabėgusias dienas, apie žodžius, kurie taip ir liko neištarti. Jis atsistojo, priėjo arčiau, bet sustojo pusiaukelėje – tarp jų vis dar kybojo nematoma siena, pastatyta iš tylos ir praradimų. „Aš niekada nep